In [0]:
print("Serverless compute + UC + Repos = working")
spark.sql("USE CATALOG retaildp")
display(spark.sql("SHOW SCHEMAS"))

In [0]:
from pyspark.sql.functions import (
    col, expr, current_timestamp, lit,
    sequence, to_date, explode, datediff,
)
from pyspark.sql.types import DoubleType

START_DATE = "2016-09-01"
END_DATE   = "2018-12-31"
RATE_START = 0.315    # BRL→USD on START_DATE
RATE_END   = 0.258    # BRL→USD on END_DATE

# Generate one row per day, linearly interpolating the rate
date_range = (
    spark.range(1).select(
        explode(sequence(
            to_date(lit(START_DATE)),
            to_date(lit(END_DATE)),
            expr("interval 1 day"),
        )).alias("rate_date")
    )
)

total_days = (
    date_range.agg({"rate_date": "max"}).collect()[0][0]
    - date_range.agg({"rate_date": "min"}).collect()[0][0]
).days

brl_rates = (
    date_range
    .withColumn("from_currency", lit("BRL"))
    .withColumn("to_currency",   lit("USD"))
    .withColumn(
        "rate",
        (
            lit(RATE_START)
            - lit(RATE_START - RATE_END)
              * datediff(col("rate_date"), to_date(lit(START_DATE)))
              / lit(total_days)
        ).cast(DoubleType())
    )
    .withColumn("_ingest_ts",   current_timestamp())
    .withColumn("_source_file", lit("manual_brl_backfill_pass3_olist"))
)

print(f"Staged {brl_rates.count():,} BRL rate rows for {START_DATE} → {END_DATE}")
brl_rates.createOrReplaceTempView("_brl_stage")

# MERGE — same idempotent pattern as the existing fx_rates loader
spark.sql("""
MERGE INTO retaildp.bronze.fx_rates AS tgt
USING _brl_stage                    AS src
   ON tgt.rate_date     = src.rate_date
  AND tgt.from_currency = src.from_currency
  AND tgt.to_currency   = src.to_currency
WHEN MATCHED THEN UPDATE SET
    tgt.rate         = src.rate,
    tgt._ingest_ts   = src._ingest_ts,
    tgt._source_file = src._source_file
WHEN NOT MATCHED THEN INSERT *
""")

In [0]:
%sql
SELECT rate_date, from_currency, to_currency, rate
FROM retaildp.bronze.fx_rates
WHERE from_currency = 'BRL' AND rate_date = DATE '2017-06-15';

In [0]:
%sql
SELECT store_no, store_name, country, local_currency
FROM retaildp.bronze.stores
WHERE store_no = 99999;